# MIS Initializer Comparison

Compares all available parameter initializers on a small MIS (Maximum Independent Set)
instance.  The same problem, mixer, and initial state are used throughout; only the
`initializer=` argument changes.

**Available initializers**

| Initializer | Strategy | Monotonic guarantee |
|---|---|---|
| `Interp()` | Linear interpolation of previous depth's angles | no |
| `LayerGrid()` | 2-D grid search over new layer; always evaluates (γ=0,β=0) | **yes** |
| `LinearRamp()` | Linearly-spaced annealing schedule | no |
| `TQA()` | Trotterised Quantum Annealing schedule | no |
| `Random(seed)` | Uniformly random; supports multistart | no |
| `FixedAngles(angles)` | User-supplied / transferred angles | no |
| `Fourier(u, v)` | Fourier (u,v) parameterisation (standard QAOA only) | no |


In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np

from qaoa import QAOA, utils, initializers
from qaoa.initialstates import MIS_MVC_InitialState
from qaoa.mixers import VertexSubsetLXMixer
from qaoa.problems import MIS_MVC_Problem
from qiskit_aer import AerSimulator


## Graph and problem


In [ ]:
# Small 6-node graph – quick to run, non-trivial MIS
G = nx.path_graph(6)
nx.draw(G, with_labels=True)
plt.title("Path graph P6")
plt.show()

problem_kind = "mis"
initial_angle = np.pi / 4

problem    = MIS_MVC_Problem(G, problem_kind=problem_kind)
init_state = MIS_MVC_InitialState(G, problem_kind=problem_kind, angle=initial_angle, phase_correct=True)
lx_mixer   = VertexSubsetLXMixer(G, problem_kind=problem_kind, multi_angle=True)

objmin, objmax = problem.objective_bounds()
print(f"Optimal MIS size: {objmax}")


## Shared settings


In [ ]:
backend    = AerSimulator()
shots      = 512
maxdepth   = 3
angles_spec = {"gamma": [0, 2 * np.pi, 10], "beta": [0, np.pi, 10]}


## Define one QAOA instance per initializer


In [ ]:
def make_qaoa(init):
    return QAOA(
        problem=problem,
        mixer=lx_mixer,
        initialstate=init_state,
        backend=backend,
        shots=shots,
        initializer=init,
    )

configs = {
    "Interp":       make_qaoa(initializers.Interp()),
    "LayerGrid":    make_qaoa(initializers.LayerGrid(
                        gamma_values=[0, 2*np.pi, 10],
                        beta_values=[0, np.pi, 10],
                    )),
    "LinearRamp":   make_qaoa(initializers.LinearRamp()),
    "TQA":          make_qaoa(initializers.TQA()),
    "Random":       make_qaoa(initializers.Random(seed=42, n_candidates=3)),
    "FixedAngles":  make_qaoa(initializers.FixedAngles(np.zeros(2))),  # all-zero start
    "Fourier":      None,   # requires n_gamma=1, n_beta=1 – LX is multi-angle, so skip
}

# Remove Fourier (not supported for multi-angle LX ansatz)
configs.pop("Fourier")
print("Initializers to compare:", list(configs.keys()))


## Run optimization for each initializer


In [ ]:
results = {}
for name, qaoa in configs.items():
    print(f"Running {name}...", end=" ", flush=True)
    qaoa.optimize(depth=maxdepth, angles=angles_spec)
    results[name] = [qaoa.get_energy(d) for d in range(1, maxdepth + 1)]
    print(f"done (depth-{maxdepth} energy: {results[name][-1]:.3f})")


## Compare: expected energy vs depth


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for name, energies in results.items():
    ax.plot(range(1, maxdepth + 1), energies, marker='o', label=name)

ax.set_xlabel("Depth p")
ax.set_ylabel("Expected energy (CVaR)")
ax.set_title("MIS on P6 – expected energy by initializer")
ax.legend(loc="lower left")
ax.grid(True)
plt.tight_layout()
plt.show()


## Compare: approximation ratio vs depth


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for name, qaoa in configs.items():
    utils.plot_ApproximationRatio(
        qaoa, maxdepth, objmin, objmax,
        label=name,
        fig=fig,
    )

ax.set_title("MIS on P6 – approximation ratio by initializer")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()


## Note on Fourier initializer

`Fourier` requires a **standard (vanilla) QAOA** ansatz with exactly one γ
and one β parameter per layer.  The LX mixer used here is multi-angle, so
`Fourier` raises a `ValueError` and is excluded from this comparison.

To use Fourier on vanilla QAOA:

```python
from qaoa import initializers
from qaoa.mixers import X

qaoa_vanilla = QAOA(
    problem=problem,
    mixer=X(),
    initialstate=...,
    initializer=initializers.Fourier(u=[0.1, 0.05], v=[0.2, 0.1]),
)
qaoa_vanilla.optimize(depth=4)
```
